# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [15]:
#df = pd.read_csv("AviationData.csv")

#df.head()

df = pd.read_csv(
    "data/AviationData.csv",
    encoding="cp1252",
    low_memory=False
)

df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [ ]:
#convert the Event.date from text to date format
df["Event.Date"] = pd.to_datetime(
    df["Event.Date"],
    errors="coerce"
)

df["Event.Date"].head()

0   1948-10-24
1   1962-07-19
2   1974-08-30
3   1977-06-19
4   1979-08-02
Name: Event.Date, dtype: datetime64[us]

In [ ]:
#This keeps only records from 1983 through 2023, inclusive.
df = df[
    (df["Event.Date"].dt.year >= 1983) &
    (df["Event.Date"].dt.year <= 2023)
].copy()

df.shape

(85289, 31)

In [ ]:
#Inspect the aircraft and accident categories
df["Aircraft.Category"].value_counts(dropna=False)

Aircraft.Category
NaN                  56566
Airplane             24441
Helicopter            3151
Glider                 456
Balloon                201
Weight-Shift           161
Gyrocraft              158
Powered Parachute       91
Ultralight              29
Unknown                 13
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

In [21]:
#This will help us distinguish accidents from incidents and other investigation types.
"""Why are we doing this? The client is specifically interested in commercial and passenger aircraft safety. 
We need to examine the available categories before deciding which records to keep, rather than accidentally removing useful aircraft."""
df["Investigation.Type"].value_counts(dropna=False)

Investigation.Type
Accident    81534
Incident     3755
Name: count, dtype: int64

In [22]:
# Check aircraft types with missing categories
df[df["Aircraft.Category"].isna()][
    ["Make", "Model"]
].head(10)

,Make,Model
3600,Piccard,AX-6
3601,Cessna,182P
3602,Cessna,182RG
3603,Cessna,182P
3604,Piper,PA-28R-200
3605,Cessna,140
3606,Balloon Works,FIREFLY 7B
3607,Cessna,340A
3608,North American,T-6G
3609,Piper,PA-24-250


In [23]:
# Count confirmed airplane accidents
airplane_accidents = df[
    (df["Aircraft.Category"] == "Airplane") &
    (df["Investigation.Type"] == "Accident")
].copy()

airplane_accidents.shape

(22901, 31)

NOTE: The dataset contains both accidents and incidents involving different aircraft categories. Since the client is interested in airplane safety, an initial subset of records explicitly classified as airplanes and accidents was created. Records with missing aircraft categories were inspected separately to avoid unnecessarily excluding potentially relevant aircraft.

In [24]:
# Examine missing aircraft categories
missing_aircraft = df[
    df["Aircraft.Category"].isna()
]

# Count investigation types
missing_aircraft["Investigation.Type"].value_counts()

Investigation.Type
Accident    54445
Incident     2121
Name: count, dtype: int64

In [ ]:
# Check professionally built aircraft
#This will tell us how many aircraft are recorded as amateur-built, professionally built, or missing that information.

df["Amateur.Built"].value_counts(dropna=False) #This means approximately 90.2% of the records are classified as not amateur-built.

Amateur.Built
No     76960
Yes     8228
NaN      101
Name: count, dtype: int64

In [26]:
# Keep non-amateur-built aircraft involved in accidents

professional_accidents = df[
    (df["Amateur.Built"] == "No") &
    (df["Investigation.Type"] == "Accident")
].copy()

# Check the remaining rows and columns
professional_accidents.shape

(73286, 31)

In [27]:
# Inspect aircraft categories after filtering

professional_accidents["Aircraft.Category"].value_counts(
    dropna=False
)

Aircraft.Category
NaN                  49485
Airplane             19937
Helicopter            2934
Glider                 434
Balloon                197
Weight-Shift           139
Powered Parachute       83
Gyrocraft               31
Ultralight              18
Unknown                  9
WSFT                     9
Blimp                    4
Powered-Lift             3
UNK                      2
Rocket                   1
Name: count, dtype: int64

The dataset was filtered to retain accident records involving aircraft identified as not amateur-built. Amateur-built aircraft and records with unknown build status were excluded from this preliminary subset to align with the client's interest in professionally manufactured aircraft. Aircraft categories will be examined further before selecting the final analysis dataset.

In [28]:
# Select records with missing aircraft categories
missing_category = professional_accidents[
    professional_accidents["Aircraft.Category"].isna()
]

# Display the most common aircraft manufacturers
missing_category["Make"].value_counts().head(15)

Make
Cessna         18328
Piper           9961
Beech           3503
Bell            1742
Grumman          905
Mooney           889
Bellanca         755
Boeing           728
Hughes           680
Robinson         656
Schweizer        534
Air Tractor      458
Aeronca          406
Maule            354
Champion         345
Name: count, dtype: int64

In [29]:
# Check common aircraft models
missing_category["Model"].value_counts().head(20)

Model
152          1904
172           877
172N          842
PA-28-140     693
172M          575
150           568
172P          520
150M          481
PA-18         472
PA-28-180     441
PA-28-161     418
PA-28-181     413
PA-38-112     386
180           383
G-164A        380
PA-18-150     375
150L          360
206B          360
G-164B        329
182           311
Name: count, dtype: int64

In [30]:
# Check aircraft manufacturers and models together

missing_category.groupby(
    ["Make", "Model"]
).size().sort_values(ascending=False).head(20)

Make     Model    
Cessna   152          1899
         172           871
         172N          836
Piper    PA-28-140     693
Cessna   172M          575
         150           566
         172P          519
         150M          481
Piper    PA-18         468
         PA-28-180     440
         PA-28-161     417
         PA-28-181     412
         PA-38-112     385
Cessna   180           382
Piper    PA-18-150     374
Cessna   150L          360
Bell     206B          359
Cessna   182           309
Grumman  G-164A        307
Piper    PA-32-300     295
dtype: int64

In [31]:
# Create a copy of our filtered dataset
clean_df = professional_accidents.copy()

# Identify known airplane models
known_airplanes = (
    (
        (clean_df["Make"] == "Cessna") &
        (clean_df["Model"].isin(["152", "172", "172N", "172M"]))
    )
    |
    (
        (clean_df["Make"] == "Piper") &
        (clean_df["Model"].isin(["PA-28-140", "PA-28-180"]))
    )
)

# Fill missing categories only for these models
clean_df.loc[
    known_airplanes & clean_df["Aircraft.Category"].isna(),
    "Aircraft.Category"
] = "Airplane"

# Check the results
clean_df["Aircraft.Category"].value_counts(dropna=False)

Aircraft.Category
NaN                  44171
Airplane             25251
Helicopter            2934
Glider                 434
Balloon                197
Weight-Shift           139
Powered Parachute       83
Gyrocraft               31
Ultralight              18
Unknown                  9
WSFT                     9
Blimp                    4
Powered-Lift             3
UNK                      2
Rocket                   1
Name: count, dtype: int64

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [32]:
# Find columns related to injuries and passengers

clean_df.columns.tolist()

['Event.Id',
 'Investigation.Type',
 'Accident.Number',
 'Event.Date',
 'Location',
 'Country',
 'Latitude',
 'Longitude',
 'Airport.Code',
 'Airport.Name',
 'Injury.Severity',
 'Aircraft.damage',
 'Aircraft.Category',
 'Registration.Number',
 'Make',
 'Model',
 'Amateur.Built',
 'Number.of.Engines',
 'Engine.Type',
 'FAR.Description',
 'Schedule',
 'Purpose.of.flight',
 'Air.carrier',
 'Total.Fatal.Injuries',
 'Total.Serious.Injuries',
 'Total.Minor.Injuries',
 'Total.Uninjured',
 'Weather.Condition',
 'Broad.phase.of.flight',
 'Report.Status',
 'Publication.Date']

In [33]:
# Select the injury-related columns

injury_columns = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

# Check the first five records
clean_df[injury_columns].head()

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
3600,0.0,1.0,0.0,1.0
3601,0.0,0.0,1.0,3.0
3602,0.0,0.0,0.0,2.0
3603,0.0,0.0,0.0,1.0
3604,0.0,0.0,2.0,0.0


In [34]:
# Count missing values in the injury columns

clean_df[injury_columns].isna().sum()

Total.Fatal.Injuries       9547
Total.Serious.Injuries    10469
Total.Minor.Injuries       9965
Total.Uninjured            4766
dtype: int64

In [35]:
# Count records with complete injury information

complete_injuries = clean_df[injury_columns].notna().all(axis=1)

print("Total records:", len(clean_df))
print("Complete injury records:", complete_injuries.sum())
print("Incomplete injury records:", (~complete_injuries).sum())


Total records: 73286
Complete injury records: 61098
Incomplete injury records: 12188


In [36]:
# Create a copy of records with complete injury information
injury_df = clean_df[complete_injuries].copy()

# Calculate total people on board
injury_df["Total.Occupants"] = injury_df[injury_columns].sum(axis=1)

# Calculate fatal and serious injuries
injury_df["Severe.Injuries"] = (
    injury_df["Total.Fatal.Injuries"] +
    injury_df["Total.Serious.Injuries"]
)

# Keep records with valid passenger totals
injury_df = injury_df[
    (injury_df[injury_columns] >= 0).all(axis=1) &
    (injury_df["Total.Occupants"] > 0)
].copy()

# Calculate proportion with fatal or serious injuries
injury_df["Severe.Injury.Rate"] = (
    injury_df["Severe.Injuries"] /
    injury_df["Total.Occupants"]
)

injury_df[[
    "Total.Occupants",
    "Severe.Injuries",
    "Severe.Injury.Rate"
]].head()

,Total.Occupants,Severe.Injuries,Severe.Injury.Rate
3600,2.0,1.0,0.5
3601,4.0,0.0,0.0
3602,2.0,0.0,0.0
3603,1.0,0.0,0.0
3604,2.0,0.0,0.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [37]:
# Examine aircraft damage categories
clean_df["Aircraft.damage"].value_counts(dropna=False)

Aircraft.damage
Substantial    55651
Destroyed      15462
NaN             1415
Minor            664
Unknown           94
Name: count, dtype: int64

In [38]:
# Create a column showing whether an aircraft was destroyed

clean_df["Destroyed"] = clean_df["Aircraft.damage"].map({
    "Destroyed": 1,
    "Substantial": 0,
    "Minor": 0
})

# Check the results
clean_df["Destroyed"].value_counts(dropna=False)

Destroyed
0.0    56315
1.0    15462
NaN     1509
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [40]:
# Check the number of missing manufacturer names
print("Missing Make values:")
print(clean_df["Make"].isna().sum())

# Count the number of unique manufacturers
print("\nUnique manufacturers:")
print(clean_df["Make"].nunique())

# Display the 20 most common manufacturers
print("\nTop 20 manufacturers:")
print(clean_df["Make"].value_counts().head(20))

Missing Make values:
20

Unique manufacturers:
2263

Top 20 manufacturers:
Make
Cessna         20590
Piper          11138
CESSNA          4820
Beech           3910
PIPER           2797
Bell            1968
Mooney          1012
BEECH           1007
Grumman          974
Robinson         911
Boeing           862
Bellanca         816
Hughes           735
Schweizer        607
Air Tractor      575
BELL             569
Aeronca          457
BOEING           446
Maule            425
Champion         412
Name: count, dtype: int64


In [39]:
# Check the most common aircraft manufacturers
clean_df["Make"].value_counts(dropna=False).head(20)

Make
Cessna         20590
Piper          11138
CESSNA          4820
Beech           3910
PIPER           2797
Bell            1968
Mooney          1012
BEECH           1007
Grumman          974
Robinson         911
Boeing           862
Bellanca         816
Hughes           735
Schweizer        607
Air Tractor      575
BELL             569
Aeronca          457
BOEING           446
Maule            425
Champion         412
Name: count, dtype: int64

In [41]:
# Count missing manufacturer names
clean_df["Make"].isna().sum()

20

In [42]:
# Count unique aircraft manufacturers
clean_df["Make"].nunique()

2263

In [43]:
# Display manufacturer names alphabetically
sorted(clean_df["Make"].dropna().unique())[:50]

['1200',
 '177MF LLC',
 '2021FX3 LLC',
 '3XTRIM',
 'A. Schleicher GMBH & Co.',
 'AAA AIRCRAFT LLC',
 'AB SPORTINE AVIACIJA',
 'AB Sportine Aviacija',
 'ADAMS BALLOONS LLC',
 'ADAMS DENNIS ALLEN',
 'AERMACCHI',
 'AERO ADVENTURE',
 'AERO AT SP ZOO',
 'AERO COMMANDER',
 'AERO SP Z O O',
 'AERO TEK INC.',
 'AERO VODOCHODY',
 'AEROFAB INC',
 'AEROFAB INC.',
 'AEROLITE',
 'AEROMOT',
 'AERONCA',
 'AEROPRAKT',
 'AEROPRAKT MANUFACTURING SP ZOO',
 'AEROPRO CZ',
 'AEROPRO CZ S R O',
 'AEROS',
 'AEROS LTD',
 'AEROS LTD/SKYRANGER AIRCRAFT',
 'AEROSPATIALE',
 'AEROSPATIALE ALENIA',
 'AEROSPOOL',
 'AEROSPORT LTD',
 'AEROSTAR',
 'AEROSTAR ACFT CORP OF TEXAS',
 'AEROSTAR AIRCRAFT CORPORATION',
 'AEROSTAR INTERNATIONAL',
 'AEROSTAR INTERNATIONAL INC',
 'AEROSTAR S A',
 'AEROTECHNIK',
 'AEROTEK',
 'AEROTEK INC',
 'AEROTRIKE',
 'AEROVODOCHODY',
 'AFICIONADO',
 'AG-CAT CORPORATION',
 'AGUSTA',
 'AGUSTA AEROSPACE CORP',
 'AGUSTA BELL',
 'AGUSTA SPA']

### Cleaning the Make Column

The following cleaning tasks was performed:

1. Identify missing manufacturer names.
2. Remove unnecessary spaces from manufacturer names.
3. Standardize capitalization to avoid duplicate manufacturer categories.
4. Inspect manufacturer names for spelling inconsistencies and alternative labels.
5. Exclude records with missing manufacturer names from make-level comparisons.
6. Retain manufacturers with at least 50 accident records for make-level analysis.


In [44]:
# Create a copy of our dataset
make_df = clean_df.copy()

# Clean manufacturer names
make_df["Make"] = (
    make_df["Make"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Replace empty names with missing values
make_df["Make"] = make_df["Make"].replace("", pd.NA)

# Check the number of unique manufacturers
print("Unique manufacturers:", make_df["Make"].nunique())

# Display the 20 most common manufacturers
make_df["Make"].value_counts().head(20)

Unique manufacturers: 1836


Make
CESSNA            25410
PIPER             13935
BEECH              4917
BELL               2537
BOEING             1308
MOONEY             1249
ROBINSON           1186
GRUMMAN            1051
BELLANCA            973
HUGHES              870
SCHWEIZER           750
AIR TRACTOR         671
AERONCA             606
MAULE               569
CHAMPION            502
STINSON             418
AERO COMMANDER      404
LUSCOMBE            390
TAYLORCRAFT         366
NORTH AMERICAN      364
Name: count, dtype: Int64

In [45]:
# Count accidents for each manufacturer
make_counts = make_df["Make"].value_counts()

# Keep manufacturers with at least 50 records
valid_makes = make_counts[make_counts >= 50].index

# Filter the dataset
make_df = make_df[
    make_df["Make"].isin(valid_makes)
].copy()

# Check the results
print("Remaining records:", len(make_df))
print("Remaining manufacturers:", make_df["Make"].nunique())

# Display the top 20 manufacturers
make_df["Make"].value_counts().head(20)

Remaining records: 66866
Remaining manufacturers: 83


Make
CESSNA            25410
PIPER             13935
BEECH              4917
BELL               2537
BOEING             1308
MOONEY             1249
ROBINSON           1186
GRUMMAN            1051
BELLANCA            973
HUGHES              870
SCHWEIZER           750
AIR TRACTOR         671
AERONCA             606
MAULE               569
CHAMPION            502
STINSON             418
AERO COMMANDER      404
LUSCOMBE            390
TAYLORCRAFT         366
NORTH AMERICAN      364
Name: count, dtype: Int64

After applying the minimum threshold of 50 accident records per manufacturer, the dataset retained 66,866 records across 83 manufacturers. This helps ensure sufficient observations for comparisons within the dataset.

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [46]:
# Count missing aircraft model names
make_df["Model"].isna().sum()

15

In [47]:
# STEP 23: INSPECT AND CLEAN THE MODEL COLUMN

# 1. Create a copy of our cleaned dataset
model_df = make_df.copy()

# 2. Check missing model names
print("Missing models before cleaning:")
print(model_df["Model"].isna().sum())

# 3. Standardize model names
model_df["Model"] = (
    model_df["Model"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 4. Replace empty model names with missing values
model_df["Model"] = model_df["Model"].replace("", pd.NA)

# 5. Remove records with missing model names
model_df = model_df.dropna(subset=["Model"]).copy()

# 6. Count remaining records
print("\nRemaining records:", len(model_df))

# 7. Count unique models
print("Unique models:", model_df["Model"].nunique())

# 8. Display the most common Make and Model combinations
print("\nTop 20 Make and Model combinations:")
print(
    model_df.groupby(["Make", "Model"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

# 9. Check whether model names appear under different Makes
model_make_counts = model_df.groupby("Model")["Make"].nunique()

shared_models = model_make_counts[model_make_counts > 1]

print("\nModels appearing under multiple manufacturers:")
print(shared_models.head(20))

# 10. Create a unique identifier for each aircraft type
model_df["Aircraft_Type"] = (
    model_df["Make"] + " - " + model_df["Model"]
)

# 11. Check the final results
print("\nFinal cleaned aircraft types:")
print(
    model_df[["Make", "Model", "Aircraft_Type"]].head(10)
)

Missing models before cleaning:
15

Remaining records: 66851
Unique models: 5400

Top 20 Make and Model combinations:
Make    Model    
CESSNA  152          2215
        172          1640
        172N         1091
PIPER   PA-28-140     862
CESSNA  172M          754
        150           745
        172P          661
        182           611
        180           594
PIPER   PA-18         550
CESSNA  150M          549
PIPER   PA-18-150     549
        PA-28-180     546
        PA-28-161     519
        PA-28-181     505
BELL    206B          485
CESSNA  150L          433
PIPER   PA-38-112     421
BEECH   A36           404
CESSNA  140           384
dtype: int64

Models appearing under multiple manufacturers:
Model
100        3
100-180    2
105        2
109        2
109A       2
112        4
112A       3
112TC      2
112TCA     3
114        2
114A       2
120        2
140        2
164        2
164B       2
170        2
180        2
188A       2
188C       2
19         2
Name: Make, dtype

In [48]:
print("Models shared by multiple manufacturers:")
print(len(shared_models))

print("\nUnique aircraft types:")
print(model_df["Aircraft_Type"].nunique())

Models shared by multiple manufacturers:
393

Unique aircraft types:
5914


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [49]:
# INSPECT OTHER COLUMNS

# Create a copy of the cleaned model dataset
final_df = model_df.copy()

# Select the columns we want to inspect
columns_to_clean = [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

# Check missing values
print("MISSING VALUES:")
print(final_df[columns_to_clean].isna().sum())

# Check the data types
print("\nDATA TYPES:")
print(final_df[columns_to_clean].dtypes)

# Check the different values in each column
for column in columns_to_clean:
    print("\n", column)
    print(final_df[column].value_counts(dropna=False).head(20))

MISSING VALUES:
Engine.Type               4292
Weather.Condition         2488
Number.of.Engines         3756
Purpose.of.flight         3592
Broad.phase.of.flight    18583
dtype: int64

DATA TYPES:
Engine.Type                  str
Weather.Condition            str
Number.of.Engines        float64
Purpose.of.flight            str
Broad.phase.of.flight        str
dtype: object

 Engine.Type
Engine.Type
Reciprocating    54401
NaN               4292
Turbo Shaft       2896
Turbo Prop        2446
Unknown           1404
Turbo Fan         1098
Turbo Jet          311
LR                   1
UNK                  1
NONE                 1
Name: count, dtype: int64

 Weather.Condition
Weather.Condition
VMC    58541
IMC     5024
NaN     2488
UNK      614
Unk      184
Name: count, dtype: int64

 Number.of.Engines
Number.of.Engines
1.0    53984
2.0     8009
NaN     3756
0.0      725
4.0      200
3.0      176
8.0        1
Name: count, dtype: int64

 Purpose.of.flight
Purpose.of.flight
Personal            

In [50]:
#CLEAN OTHER COLUMNS

# 1. Clean Engine.Type
final_df["Engine.Type"] = (
    final_df["Engine.Type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 2. Clean Weather.Condition
final_df["Weather.Condition"] = (
    final_df["Weather.Condition"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 3. Clean Number.of.Engines
final_df["Number.of.Engines"] = pd.to_numeric(
    final_df["Number.of.Engines"],
    errors="coerce"
)

# Replace impossible engine counts with missing values
final_df.loc[
    final_df["Number.of.Engines"] <= 0,
    "Number.of.Engines"
] = pd.NA

# 4. Clean Purpose.of.flight
final_df["Purpose.of.flight"] = (
    final_df["Purpose.of.flight"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 5. Clean Broad.phase.of.flight
final_df["Broad.phase.of.flight"] = (
    final_df["Broad.phase.of.flight"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 6. Replace empty strings with missing values
text_columns = [
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

final_df[text_columns] = final_df[text_columns].replace(
    "", pd.NA
)

# 7. Verify the cleaning
print("Missing values after cleaning:")
print(final_df[columns_to_clean].isna().sum())

print("\nDataset shape:")
print(final_df.shape)

print("\nFirst five records:")
display(final_df[columns_to_clean].head())

Missing values after cleaning:
Engine.Type               4292
Weather.Condition         2488
Number.of.Engines         4481
Purpose.of.flight         3592
Broad.phase.of.flight    18583
dtype: int64

Dataset shape:
(66851, 33)

First five records:


,Engine.Type,Weather.Condition,Number.of.Engines,Purpose.of.flight,Broad.phase.of.flight
3601,RECIPROCATING,VMC,1.0,PERSONAL,APPROACH
3602,RECIPROCATING,VMC,1.0,PERSONAL,LANDING
3603,RECIPROCATING,VMC,1.0,PERSONAL,TAKEOFF
3604,RECIPROCATING,VMC,1.0,PERSONAL,APPROACH
3605,RECIPROCATING,VMC,1.0,INSTRUCTIONAL,LANDING


In [51]:
# Investigate invalid engine counts
model_df.loc[
    model_df["Number.of.Engines"].notna() &
    (model_df["Number.of.Engines"] <= 0),
    "Number.of.Engines"
].value_counts()


Number.of.Engines
0.0    725
Name: count, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [52]:
#CHECK MISSING VALUES

# Calculate missing percentages
missing_percent = (
    final_df.isna().mean() * 100
).round(2)

# Sort from highest to lowest
missing_percent = missing_percent.sort_values(
    ascending=False
)

# Display results
print("Missing values by percentage:")
print(missing_percent)

Missing values by percentage:
Schedule                  87.32
Air.carrier               82.93
FAR.Description           70.28
Longitude                 62.64
Latitude                  62.63
Aircraft.Category         62.18
Airport.Code              43.25
Airport.Name              40.58
Broad.phase.of.flight     27.80
Publication.Date          17.21
Total.Serious.Injuries    14.34
Total.Minor.Injuries      13.58
Total.Fatal.Injuries      13.06
Number.of.Engines          6.70
Engine.Type                6.42
Total.Uninjured            6.39
Report.Status              5.98
Purpose.of.flight          5.37
Weather.Condition          3.72
Destroyed                  1.85
Aircraft.damage            1.72
Registration.Number        1.32
Injury.Severity            0.40
Country                    0.26
Location                   0.05
Event.Id                   0.00
Amateur.Built              0.00
Investigation.Type         0.00
Model                      0.00
Make                       0.00
Event.Date

In [53]:
# Identify columns with more than 50% missing values

high_missing = missing_percent[
    missing_percent > 50
]

print("Columns with over 50% missing values:")
print(high_missing)

Columns with over 50% missing values:
Schedule             87.32
Air.carrier          82.93
FAR.Description      70.28
Longitude            62.64
Latitude             62.63
Aircraft.Category    62.18
dtype: float64


In [54]:
# REMOVE UNNECESSARY COLUMNS

# Create a list of columns to remove
columns_to_drop = [
    "Schedule",
    "Air.carrier",
    "FAR.Description",
    "Longitude",
    "Latitude"
]

# Remove the selected columns
final_df = final_df.drop(columns=columns_to_drop)

# Check the remaining dataset
print("Remaining rows and columns:")
print(final_df.shape)

# Display remaining columns
print("\nRemaining columns:")
print(final_df.columns.tolist())

Remaining rows and columns:
(66851, 28)

Remaining columns:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Destroyed', 'Aircraft_Type']


### Column Removal

The dataset was examined to identify columns containing a high percentage of missing values. A threshold of 50% was used to identify columns for further evaluation.

Five columns were selected for removal: `Schedule`, `Air.carrier`, `FAR.Description`, `Longitude`, and `Latitude`.

These columns contained substantial missing information and were not essential to the primary analysis of aircraft destruction and passenger injury outcomes.

Although `Aircraft.Category` contained 62.18% missing values, it was retained temporarily because it may help identify airplane records and exclude other aircraft categories.

Columns related to aircraft damage, injuries, manufacturers, models, weather conditions, and flight characteristics were retained because they are relevant to the client's safety analysis.

In [55]:
# Confirm missing values after column removal

print("Dataset shape:")
print(final_df.shape)

print("\nRemaining missing percentages:")
print(
    (final_df.isna().mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)

Dataset shape:
(66851, 28)

Remaining missing percentages:
Aircraft.Category         62.18
Airport.Code              43.25
Airport.Name              40.58
Broad.phase.of.flight     27.80
Publication.Date          17.21
Total.Serious.Injuries    14.34
Total.Minor.Injuries      13.58
Total.Fatal.Injuries      13.06
Number.of.Engines          6.70
Engine.Type                6.42
Total.Uninjured            6.39
Report.Status              5.98
Purpose.of.flight          5.37
Weather.Condition          3.72
Destroyed                  1.85
Aircraft.damage            1.72
Registration.Number        1.32
Injury.Severity            0.40
Country                    0.26
Location                   0.05
Event.Id                   0.00
Amateur.Built              0.00
Investigation.Type         0.00
Model                      0.00
Make                       0.00
Event.Date                 0.00
Accident.Number            0.00
Aircraft_Type              0.00
dtype: float64


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [56]:
# SAVE CLEANED DATASET

# Save the cleaned data as a CSV file
final_df.to_csv(
    "data/AviationData_Cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

# Confirm the dataset size
print("Rows:", final_df.shape[0])
print("Columns:", final_df.shape[1])

Cleaned dataset saved successfully!
Rows: 66851
Columns: 28


### Saving the Cleaned Dataset

After completing the initial data cleaning process, the dataset was exported to a CSV file named `AviationData_Cleaned.csv`.

The dataset contains 66,851 accident records and 28 columns. The cleaning process included standardizing aircraft manufacturer and model names, constructing injury and aircraft destruction metrics, cleaning selected categorical variables, and removing columns with substantial missing information that were not essential to the analysis.

The cleaned dataset will be used for further filtering, exploratory data analysis, visualization, and aircraft safety comparisons.